<a href="https://colab.research.google.com/github/saikirankesoju/Quantum_Computing/blob/main/Product_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# STEP 1: INSTALL REQUIRED LIBRARIES (RUN ONCE IN COLAB)
# =====================================================================
!pip install -q qiskit qiskit-aer pylatexenc matplotlib scipy numpy

# =====================================================================
# STEP 2: HYBRID QUANTUM OPTIMIZATION SYSTEM (PRODUCT 3)
# Student: Kesoju Saikiran | Roll: 2303A51539 | Guide: DURGESH NANDAN
# =====================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import RealAmplitudes, QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

print("Qiskit version successfully loaded!")

# ---------------------------------------------------------------------
# A. BUILD REALISTIC NISQ NOISE MODEL
# ---------------------------------------------------------------------
def build_noise_model(p1=0.005, p2=0.02, p_ro=0.04):
    noise_model = NoiseModel()
    # 1-qubit depolarizing gate error (0.5%)
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ['rx', 'ry', 'rz', 'h', 'x', 'sx'])
    # 2-qubit depolarizing gate error (2.0%)
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ['cx', 'cz'])
    # Readout / SPAM detector bit-flip error (4.0%)
    noise_model.add_all_qubit_readout_error(ReadoutError([[1 - p_ro, p_ro], [p_ro, 1 - p_ro]]))
    return noise_model

# ---------------------------------------------------------------------
# B. PROBLEM DEFINITIONS (MAX-CUT & ISING)
# ---------------------------------------------------------------------
def get_maxcut_hamiltonian(edges, num_nodes):
    pauli_list = []
    for u, v in edges:
        z_str = ['I'] * num_nodes
        z_str[num_nodes - 1 - u] = 'Z'
        z_str[num_nodes - 1 - v] = 'Z'
        pauli_list.append((''.join(z_str), 0.5))
    return SparsePauliOp.from_list(pauli_list)

def maxcut_brute_force(edges, num_nodes):
    best_cut = -1
    best_strings = []
    for i in range(2**num_nodes):
        b = format(i, f'0{num_nodes}b')
        cut = sum(1 for u, v in edges if b[num_nodes - 1 - u] != b[num_nodes - 1 - v])
        if cut > best_cut:
            best_cut = cut
            best_strings = [b]
        elif cut == best_cut:
            best_strings.append(b)
    return best_cut, best_strings

# ---------------------------------------------------------------------
# C. READOUT CALIBRATION MATRIX & EXPECTATION EVALUATION
# ---------------------------------------------------------------------
def build_readout_calibration_matrix(num_qubits, simulator, shots=2048):
    dim = 2**num_qubits
    M = np.zeros((dim, dim))
    for i in range(dim):
        b = format(i, f'0{num_qubits}b')
        qc = QuantumCircuit(num_qubits)
        for q, bit in enumerate(reversed(b)):
            if bit == '1':
                qc.x(q)
        qc.measure_all()
        tqc = transpile(qc, simulator, optimization_level=1)
        counts = simulator.run(tqc, shots=shots).result().get_counts()
        for meas_str, cnt in counts.items():
            M[int(meas_str, 2), i] = cnt / shots
    return M

def evaluate_circuit_energy(circuit, hamiltonian, simulator, shots=2048, mitigation_matrix=None):
    tqc = transpile(circuit, simulator, optimization_level=1)
    counts = simulator.run(tqc, shots=shots).result().get_counts()

    # Apply Readout Error Mitigation via Matrix Inversion (M^-1)
    if mitigation_matrix is not None:
        dim = 2**circuit.num_qubits
        p_vec = np.zeros(dim)
        for k, v in counts.items():
            p_vec[int(k, 2)] = v / shots
        inv_m = np.linalg.inv(mitigation_matrix)
        p_mit = np.clip(np.dot(inv_m, p_vec), 0, None)
        if np.sum(p_mit) > 0:
            p_mit /= np.sum(p_mit)
        counts = {format(i, f'0{circuit.num_qubits}b'): p_mit[i] * shots for i in range(dim)}

    # Compute expectation value <H>
    total_energy = 0.0
    for term, coeff in hamiltonian.to_list():
        term_val = sum((-1 if sum(int(b[i]) for i, ch in enumerate(term) if ch == 'Z') % 2 else 1) * cnt
                       for b, cnt in counts.items()) / shots
        total_energy += coeff.real * term_val
    return total_energy, counts

# ---------------------------------------------------------------------
# D. RUN BENCHMARKS
# ---------------------------------------------------------------------
sim_ideal = AerSimulator()
sim_noisy = AerSimulator(noise_model=build_noise_model())

print("=" * 70)
print("PRODUCT 3: HYBRID QUANTUM OPTIMIZATION SYSTEM BENCHMARK")
print("=" * 70)

# 1. QAOA 4-Node Max-Cut
edges_4 = [(0, 1), (1, 2), (2, 3), (3, 0)]
num_nodes = 4
best_cut, opt_strings = maxcut_brute_force(edges_4, num_nodes)
hamiltonian_qaoa = get_maxcut_hamiltonian(edges_4, num_nodes)
ansatz_qaoa = QAOAAnsatz(cost_operator=hamiltonian_qaoa, reps=1)
cal_matrix = build_readout_calibration_matrix(num_nodes, sim_noisy, shots=2048)

def cost_qaoa(params, sim, mit=None):
    qc = ansatz_qaoa.assign_parameters(params)
    qc.measure_all()
    e, _ = evaluate_circuit_energy(qc, hamiltonian_qaoa, sim, shots=1024, mitigation_matrix=mit)
    return e

init_p = np.array([0.5, 0.5])
res_ideal = minimize(cost_qaoa, init_p, args=(sim_ideal, None), method='COBYLA', options={'maxiter': 25})
res_noisy = minimize(cost_qaoa, init_p, args=(sim_noisy, None), method='COBYLA', options={'maxiter': 25})

qc_id = ansatz_qaoa.assign_parameters(res_ideal.x); qc_id.measure_all()
qc_ns = ansatz_qaoa.assign_parameters(res_noisy.x); qc_ns.measure_all()

_, c_ideal = evaluate_circuit_energy(qc_id, hamiltonian_qaoa, sim_ideal, shots=2048)
_, c_noisy = evaluate_circuit_energy(qc_ns, hamiltonian_qaoa, sim_noisy, shots=2048)
_, c_mit   = evaluate_circuit_energy(qc_ns, hamiltonian_qaoa, sim_noisy, shots=2048, mitigation_matrix=cal_matrix)

def get_metrics(counts):
    tot = sum(counts.values())
    w_cut = sum(sum(1 for u, v in edges_4 if b[num_nodes - 1 - u] != b[num_nodes - 1 - v]) * c for b, c in counts.items())
    opt_p = sum(c for b, c in counts.items() if b in opt_strings) / tot
    return w_cut / tot, (w_cut / tot) / best_cut, opt_p

cut_id, ar_id, pr_id = get_metrics(c_ideal)
cut_ns, ar_ns, pr_ns = get_metrics(c_noisy)
cut_mit, ar_mit, pr_mit = get_metrics(c_mit)

print("\n[QAOA Max-Cut Results (4-Node Ring Graph)]")
print(f"  Exact Maximum Cut Value      : {best_cut}")
print(f"  Optimal Bitstrings           : {opt_strings}")
print(f"  Ideal Execution              : Expected Cut={cut_id:.3f} | Approx Ratio={ar_id:.3f} | Opt Prob={pr_id*100:.2f}%")
print(f"  Noisy Execution (Unmitigated): Expected Cut={cut_ns:.3f} | Approx Ratio={ar_ns:.3f} | Opt Prob={pr_ns*100:.2f}%")
print(f"  Error-Mitigated Execution    : Expected Cut={cut_mit:.3f} | Approx Ratio={ar_mit:.3f} | Opt Prob={pr_mit*100:.2f}%")
print("=" * 70)

Qiskit version successfully loaded!
PRODUCT 3: HYBRID QUANTUM OPTIMIZATION SYSTEM BENCHMARK

[QAOA Max-Cut Results (4-Node Ring Graph)]
  Exact Maximum Cut Value      : 4
  Optimal Bitstrings           : ['0101', '1010']
  Ideal Execution              : Expected Cut=2.956 | Approx Ratio=0.739 | Opt Prob=51.76%
  Noisy Execution (Unmitigated): Expected Cut=2.706 | Approx Ratio=0.677 | Opt Prob=40.38%
  Error-Mitigated Execution    : Expected Cut=2.871 | Approx Ratio=0.718 | Opt Prob=47.82%
